# Where does MegaDescriptor look? Similarity heatmaps

Like the attention maps in lecture ("Show, Attend and Tell"), these figures color each frame by **which regions made the baseline match a validation video to a training video**.

**Why not raw attention weights?** MegaDescriptor is a Swin transformer: it has no `[CLS]` token and its self-attention only looks inside small local windows, so there is no single attention map to read off like in a plain ViT.

**What we use instead (exact for our baseline):**
- The last Swin stage gives a **7 x 7 grid** of 768-number patch features per frame.
- The frame embedding is the **average of those 49 patch features**; the video embedding is the normalized average of its 20 frame embeddings.
- So the cosine similarity splits exactly into one number per patch: `patch feature . unit embedding of the other video`. Adding up every patch of every frame gives back the similarity.

Bright = the region pushed the two videos together. Negative scores are clipped to 0 (dark).

We look at **5 correct** validation videos (5 different bears) and **all 24 wrong** ones. The test split stays held out, so it is never used here.

Needs the frames of these videos in `data/PolarBearVidID/` (see README).

## 0. Imports and setup

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repo folder (one level up from notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

import heatmaps
from preprocess import load_feature_extractor

## 1. Rerun the baseline and keep the details of every validation video

One row per validation video: the predicted bear, the matched training video and its similarity, and the closest training video of the **true** bear.

In [ ]:
results, seq_to_emb = heatmaps.match_query_videos(query_split="validation")
results.head()

Accuracy check: this should be the same 88.84% (191/215) as `core_objective.py`.

In [ ]:
results["correct"].mean() * 100, results["correct"].sum(), len(results)

## 2. Pick the examples

5 correct validation videos, each from a different bear (fixed seed), and every wrong one.

In [ ]:
successes, failures = heatmaps.pick_examples(results, n_success=5, seed=0)
successes

In [ ]:
failures

How close were the failures? `match_sim` is the wrong video that won, `true_best_sim` is the best video of the right bear. A small gap means a near miss.

In [ ]:
gap = (failures["match_sim"] - failures["true_best_sim"]).rename("gap")
failures[["sequence", "identity", "predicted"]].join(gap).sort_values("gap")

## 3. Load the frame table and the model

In [ ]:
frames, embeddings = heatmaps.load_frames()
frames.head()

In [ ]:
model, transform = load_feature_extractor(device="cpu")
type(model).__name__

## 4. Check the math on one frame

The frame embedding we saved should equal the average of the 7 x 7 patch grid (difference about 0).

In [ ]:
row = frames[frames["sequence"] == successes["sequence"].iloc[0]].iloc[0]
patches = heatmaps.patch_features(model, transform, [row["member"]])
patches.shape

In [ ]:
np.abs(patches[0].mean(axis=(0, 1)) - embeddings[row.name]).max()

## 5. Correct predictions

Each figure:
- **Row 1**: the validation video, colored by what matched the predicted training video.
- **Row 2**: the matched training video, colored by what matched the validation video.
- The % above a frame is that frame's share of the video's total similarity (8 of the 20 frames are shown).

All rows of a figure share one color scale.

In [ ]:
success_paths = []
for _, example in successes.iterrows():
    fig, path = heatmaps.plot_example(example, frames, embeddings, seq_to_emb, model, transform)
    plt.close(fig)
    success_paths.append(path)
success_paths

In [ ]:
for path in success_paths:
    display(ShowImage(path))

## 6. Wrong predictions

Same rows, plus **row 3**: the closest training video of the true bear, colored by what matched the validation video. Compare rows 2 and 3 to see what the wrong bear had in common with the validation video.

In [ ]:
failure_paths = []
for _, example in failures.iterrows():
    fig, path = heatmaps.plot_example(example, frames, embeddings, seq_to_emb, model, transform)
    plt.close(fig)
    failure_paths.append(path)
len(failure_paths)

In [ ]:
for path in failure_paths:
    display(ShowImage(path))

All figures are saved in `results/heatmaps/`.